In [2]:
version = "REPLACE_PACKAGE_VERSION"

# Experiment Design and Analysis
## School of Information, University of Michigan

## Week 3: 
- 1. Power & Sample Sizes
- 2. Randomization - Blocking & Clustering
- 3. Differences-in-Differences

## Assignment Overview
### The objective of this assignment is to:

- Applying theory of experiment design and knowledge of analysis techniques to real experiment data.


### The total score of this assignment will be 18 points


### Resources:
- StatsModels and Scipy.stats
    - We recommend using two python libraries called [StatsModels](https://www.statsmodels.org/stable/index.html) and [scipy.stats](https://docs.scipy.org/doc/scipy/reference/tutorial/stats.html) for data analysis

- Datasets used for this assignment:
    - MovieLens Data: [assignment3_data.csv](assignment3_data.csv)
        - Source for dataset: [Chen, Y. et al. Social Comparisons and Contributions to Online Communities: A Field Experiment on MovieLens. (2010).](https://www-jstor-org.proxy.lib.umich.edu/stable/27871259)
    

In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
import math as math
from scipy import stats
from statsmodels.stats.power import TTestIndPower
from mads.lib.path import assets
#you may or may not use all of the above libraries, and that is OK!

file = assets.find("assignment3_data.csv")
movie_data = pd.read_csv(file) #Data for this assignment

In [4]:
#uncomment the below line to view readme files for this dataset (includes explanation of variable names)
file_readme = assets.find("assignment3_data_readme.md")
!cat $file_readme

#uncomment the below line to view snippet of csv file
movie_data.head()

### Assignment Topic: Data analysis of a field experiment on MovieLens

### Background:
We upload data files from a field experiment conducted on MovieLens, which was mentioned in lecture this week.

### Data:

The MovieLens data ([assignment3_data.csv](assignment3_data.csv)) has the following variables:

- userid: user ID

- expcondition: experiment condtion: control group or treatment group

- compare_w_median: the total number of movie a user has rated compared with the group median. "1": above median, "0": about median, "-1": below median.

- ratings_lifetime: the total number of movie a user has rated before experiment

- edu_year: length of education in years

- pre_rating: the number of movie a user has rated in one month before the experimental intervention

- post_rating: the number of movie a user has rated in one month after the experimental intervention

- active: A user is active if she rated movies, updated the database, or invited a buddy during the two-month period of d

,userid,expcondition,userage,compare_w_median,ratings_lifetime,pre_rating,post_rating,active,male,weeks,control
0,42126,control,old,1,1300,0,0,0,0,496,1
1,47947,control,old,0,765,0,38,1,0,361,1
2,49034,control,old,-1,313,0,41,1,0,347,1
3,51898,conformity,old,0,898,0,8,1,1,301,0
4,52797,conformity,old,0,885,0,33,1,0,300,0


## Introduction
In `movie_data`, you will find a dataframe containing a portion of the data from MovieLens experiment. To simply this assignment, you will only find one treatment condition where the experimenters tested the impact of social influence on moving ratings. This treatment was administrated through sending a tailored email that emphasized social influence. In contrast, subjects in the control received a plain version of email.

## Part A (6 points)

First, we should check if our sample is relatively balanced across our treatment and control groups. Test the following hypotheses using a t-test:

1. The number of ratings in the month before the intervention (`pre_rating`) are balanced between the treatment and control groups. (3 points)

**Round any calculations to the hundredth decimal. Do not use percentages.**

### Solution notes for Part A.1
This function splits the sample into control and treatment groups, compares **pre_rating** with a two-sample t-test, and returns the one-row summary table in the exact format required by the autograder.

In [ ]:
def pre_ratings(provided_data):

    """
    Compare the mean number of pre-intervention ratings between the control and
    treatment groups using an independent-samples t-test.

    Returns a one-row DataFrame named
    'Difference in Means between Pre-Rating Groups'.
    """
    # Work on a copy so the original input dataframe is not modified in place.
    df = provided_data.copy()

    # Split the pre_rating variable into the two experimental groups.
    control_vals = df[df['control'] == 1]['pre_rating']
    treatment_vals = df[df['control'] == 0]['pre_rating']

    # Run a standard two-sample t-test to compare group means.
    t_stat, p_val = stats.ttest_ind(control_vals, treatment_vals)

    # Build the required one-row output table in the exact column order
    # expected by the notebook tests.
    pr_df = pd.DataFrame([{
        'avg control': round(control_vals.mean(), 2),
        'avg treatment': round(treatment_vals.mean(), 2),
        't-statistic': round(float(t_stat), 2),
        'p-value': round(float(p_val), 2)
    }])

    # Set the DataFrame name exactly as required by the assignment.
    pr_df.name = "Difference in Means between Pre-Rating Groups"

    return pr_df

Your function should return a named dataframe with each of the variables and their completed statistics. Check that it does:

In [ ]:
pre_ratings(movie_data)

In [ ]:
assert pre_ratings(movie_data).name == "Difference in Means between Pre-Rating Groups"
df_columns = ['avg control', 'avg treatment', 't-statistic', 'p-value']
for index, title in enumerate(pre_ratings(movie_data).columns):
    assert title == df_columns[index]

In [ ]:
"""Checking avg_control and avg_treatment values"""
# Hidden tests

In [ ]:
"""checking your t-statistic and p-value are correct"""
# Hidden tests

2. Test that the gender composition (variable 'male') is similar between the treatment and control groups. (3 points)

**Round any calculations to the hundredth decimal. Do not use percentages.**

### Solution notes for Part A.2
This function treats **male** as a binary indicator, compares the average value across groups, and reports the t-test summary used to check balance on gender.

In [ ]:
def male_gender_comp(provided_data):

    """
    Compare the proportion of males across the control and treatment groups
    using an independent-samples t-test.

    Returns a one-row DataFrame named 'Difference in Means of Males'.
    """
    # Copy the input to avoid changing the original dataframe.
    df = provided_data.copy()

    # Separate the male indicator for each experimental group.
    gender_control = df[df['control'] == 1]['male']
    gender_treatment = df[df['control'] == 0]['male']

    # Test whether the average male indicator differs across groups.
    t_stat, p_val = stats.ttest_ind(gender_control, gender_treatment)

    # Assemble the output in the exact format requested by the assignment.
    male_df = pd.DataFrame([{
        'avg control': round(gender_control.mean(), 2),
        'avg treatment': round(gender_treatment.mean(), 2),
        't-statistic': round(float(t_stat), 2),
        'p-value': round(float(p_val), 2)
    }])

    # Apply the required DataFrame name.
    male_df.name = "Difference in Means of Males"

    return male_df

Your function should return a named dataframe with each of the variables and their completed statistics. Check that it does:

In [ ]:
male_gender_comp(movie_data)

In [ ]:
"""Checking the t-statistic and p-value in your dataframe are correct"""
assert next(iter(male_gender_comp(movie_data)['t-statistic'])) == -0.49
assert next(iter(male_gender_comp(movie_data)['p-value'])) == 0.62

In [ ]:
"""Part A #2: Checking your dataframe is named, and your columns are in order"""
# Hidden tests

In [ ]:
"""Part A #2: Checking your avg control and avg treatment values are correct"""
# Hidden tests

## Part B (6 points)

From the MovieLens experiment, we know that we want to estimate the impact of social influence on moving ratings on the MovieLens platform. Let’s estimate this by using difference-in-differences to examine the effects of post_rating for the treatment and control group.

1. Create a new variable, delta, in the dataframe and output the dataframe. Delta should show the difference in pre_rating and post_rating (calculate using post_rating – pre_rating). (2 points)

### Solution notes for Part B
Here we create the **delta** outcome, defined as post-period ratings minus pre-period ratings, and keep only the columns requested by the assignment.

In [ ]:
def delta_ratings(provided_data):

    """
    Create a new dataframe that includes the assignment's requested columns and
    a delta variable defined as post_rating - pre_rating.

    Returns a DataFrame named 'Delta in Ratings'.
    """
    # Copy the dataframe so the original data remains unchanged.
    df = provided_data.copy()

    # Compute each user's change in ratings from before to after treatment.
    df['delta'] = df['post_rating'] - df['pre_rating']

    # Keep only the columns requested by the assignment, in the required order.
    delta_ratings_df = df[[
        'userid', 'compare_w_median', 'pre_rating',
        'post_rating', 'delta', 'control'
    ]].copy()

    # Name the output DataFrame exactly as required.
    delta_ratings_df.name = "Delta in Ratings"

    return delta_ratings_df

Your function should return a named dataframe with each of the variables and their values. Check that it does:

In [ ]:
delta_ratings(movie_data).head()

In [ ]:
"""checking you have a named dataframe"""
assert delta_ratings(movie_data).name == "Delta in Ratings"

In [ ]:
"""checking your column names and orders"""
# Hidden tests

2. Use an ordinary least squares regression model to explore the average treatment-effect on delta. Using a t-test, what is the significance using the t-statistic and p-value of this effect? (4 points)

**Round any calculations to the hundredth decimal. Do not use percentages.**

### Solution notes for Part C.1
This part estimates the average treatment effect by regressing **delta** on the **control** indicator with OLS and then extracting the t-statistic and p-value for that coefficient.

In [ ]:
def ate_delta_avg(provided_data):

    """ 
    Estimate the average treatment effect on delta by regressing delta on the
    control indicator using OLS.

    Returns a one-row DataFrame named 'Average Treatment Effect on Delta'
    containing the t-statistic and p-value for the control coefficient.
    """
    # Work on a copy so no in-place changes affect later notebook cells.
    df = provided_data.copy()

    # Create delta if it is not already present in the input dataframe.
    if 'delta' not in df.columns:
        df['delta'] = df['post_rating'] - df['pre_rating']

    # Define the regression outcome and predictors.
    y = df['delta']
    X = sm.add_constant(df['control'])

    # Fit the OLS model.
    model = sm.OLS(y, X).fit()

    # Extract the t-statistic and p-value for the treatment/control indicator.
    test_res = model.t_test('control')
    t_test = test_res.tvalue[0][0]
    p_value = test_res.pvalue

    # Return the results in the exact structure expected by the tests.
    did_df = pd.DataFrame([{
        't-statistic': round(float(t_test), 2),
        'p-value': round(float(p_value), 2)
    }])
    did_df.name = "Average Treatment Effect on Delta"

    return did_df

Your function should return a named dataframe with the correct values. Check that it does:

In [ ]:
ate_delta_avg(movie_data) #again, if you get a deprecation warning, that is fine.

In [ ]:
"""Part B #2: Checking your dataframe is named, and your columns are in order"""
# Hidden tests

In [ ]:
"""checking your t-statistic and p-value are correct"""
# Hidden tests

## Part C (8 points)

What if we break this comparison down by group, specifically the total number of ratings users complete compared with the median ratings (compare_w_median)?

1. Output the t-statistics and p-values for the average treatment-effect on delta across median ratings (where ```compare_w_median``` == -1, where ```compare_w_median``` == 0, and where ```compare_w_median``` == 1). (8 points)

### Solution notes for Part C.2
This repeats the Part C regression within each **compare_w_median** subgroup so the treatment effect can be evaluated below, at, and above the median.

In [ ]:
def ate_delta_median_values(provided_data):
    """ 
    Estimate the average treatment effect on delta separately for users below
    the median, at the median, and above the median.

    Returns a DataFrame named 'Average Treatment Effect across Median Scores'
    with index values: 'below median', 'at median', and 'abv median'.
    """
    # Copy the data to avoid modifying the original dataframe.
    df = provided_data.copy()

    # Create delta if it does not already exist.
    if 'delta' not in df.columns:
        df['delta'] = df['post_rating'] - df['pre_rating']

    # Map the coded compare_w_median values to the required index labels.
    index_map = {
        -1: 'below median',
         0: 'at median',
         1: 'abv median'
    }

    rows = []

    # Loop in the exact order expected by the visible and hidden tests.
    for key in [-1, 0, 1]:
        # Restrict the data to one median-comparison subgroup.
        sub = df[df['compare_w_median'] == key]

        # Fit the subgroup-specific OLS regression of delta on control.
        y = sub['delta']
        X = sm.add_constant(sub['control'])
        model = sm.OLS(y, X).fit()

        # Pull the test statistics for the control coefficient.
        test_res = model.t_test('control')
        rows.append({
            'group': index_map[key],
            't-statistic': round(float(test_res.tvalue[0][0]), 2),
            'p-value': round(float(test_res.pvalue), 2)
        })

    # Convert the collected subgroup results into the required output table.
    did_med_df = pd.DataFrame(rows).set_index('group')
    did_med_df.name = 'Average Treatment Effect across Median Scores'

    return did_med_df

Your function should return a named and indexed dataframe with the correct values. Check that it does:

In [ ]:
ate_delta_median_values(delta_ratings(movie_data))

In [ ]:
"""checking your dataframe is named, your columns are in order, and you have a dataframe index"""
assert ate_delta_median_values(delta_ratings(movie_data)).name == 'Average Treatment Effect across Median Scores'
check_col = iter(ate_delta_median_values(delta_ratings(movie_data)).columns)
check_ind = iter(ate_delta_median_values(delta_ratings(movie_data)).index)
assert next(check_ind) == 'below median'
assert next(check_ind) == 'at median'
assert next(check_ind) == 'abv median'
assert next(check_col) == 't-statistic'
assert next(check_col) == 'p-value'

In [ ]:
"""checking your below-median t-statistic and p-value are correct"""
# Hidden tests

In [ ]:
"""checking your at-median t-statistic and p-value are correct"""
# Hidden tests

In [ ]:
"""checking your abv-median t-statistic and p-value are correct"""
# Hidden tests

## Part D (5 points)

Based on the `post_rating` observations, perform a sample-size calculation to determine the minimum number of total subjects (treatment subjects plus control subjects) that are needed for detecting difference in means between the treament and the control groups. Use $\alpha=0.05$ and $\beta = 0.1$.

The `solve_power` method of the `TTestIndPower` class provided by `statsmodels` can be used to solve this problem. You may want to carefully read its [documentation] (https://www.statsmodels.org/stable/generated/statsmodels.stats.power.TTestIndPower.solve_power.html#statsmodels.stats.power.TTestIndPower.solve_power) to understand how to use it. Use `TTestIndPower().solve_power` to call the function. Make sure to round up the number of observations by using math.ceil(....).

### Solution notes for Part D
This section computes the control and treatment means, the pooled standard deviation, Cohen’s *d*, and the minimum required total sample size using the observed treatment-control allocation ratio.

In [ ]:
def power_calc(provided_data):
    """
    Compute the ingredients for a two-sample power analysis using post_rating.

    Returns a named Series, 'Power Analysis', containing:
      - ctrl_mean
      - trtm_mean
      - pooled_std
      - num_obs
    """
    # Copy the data so the original dataframe is preserved.
    df = provided_data.copy()

    # Split post-intervention ratings into control and treatment groups.
    control_post = df[df['control'] == 1]['post_rating']
    treatment_post = df[df['control'] == 0]['post_rating']

    # Compute the group means used in the power analysis summary.
    ctrl_mean = control_post.mean()
    trtm_mean = treatment_post.mean()

    # Get sample sizes and sample standard deviations for each group.
    n_ctrl = len(control_post)
    n_trtm = len(treatment_post)
    s_ctrl = control_post.std(ddof=1)
    s_trtm = treatment_post.std(ddof=1)

    # Compute the pooled standard deviation for the two-group design.
    pooled_std = np.sqrt(
        (((n_ctrl - 1) * (s_ctrl ** 2)) + ((n_trtm - 1) * (s_trtm ** 2))) /
        (n_ctrl + n_trtm - 2)
    )

    # Convert the observed mean difference into Cohen's d.
    effect_size = abs(trtm_mean - ctrl_mean) / pooled_std

    # Use the observed allocation ratio rather than forcing a balanced design.
    ratio = n_trtm / n_ctrl

    # solve_power returns the required sample size for group 1.
    analysis = TTestIndPower()
    n_ctrl_req = analysis.solve_power(
        effect_size=effect_size,
        alpha=0.05,
        power=0.9,
        ratio=ratio,
        alternative='two-sided'
    )

    # Round each group's requirement up separately, then total them.
    n_ctrl_req = math.ceil(n_ctrl_req)
    n_trtm_req = math.ceil(n_ctrl_req * ratio)
    num_obs = n_ctrl_req + n_trtm_req

    # Build the named Series in the exact order expected by the tests.
    power_series = pd.Series(
        [ctrl_mean, trtm_mean, pooled_std, num_obs],
        index=['ctrl_mean', 'trtm_mean', 'pooled_std', 'num_obs'],
        name='Power Analysis'
    )

    return power_series

In [ ]:
# Check your result
power_calc(movie_data)

In [ ]:
# Visible tests

stu_ans = power_calc(movie_data)

assert isinstance(stu_ans, pd.Series), "Part D: Your function should return a pd.Series. "
assert stu_ans.name == "Power Analysis", "Part D: Your Series should be named correctly. "
assert list(stu_ans.index) == ['ctrl_mean', 'trtm_mean', 'pooled_std', 'num_obs'], "Part D: Your Series should have the correct indices. "

del stu_ans

In [ ]:
# Hidden tests
